# Analysis of CPU and RAM Utilization

This notebook calculates stats and generates plots for CPU and RAM utilization based on the processes previously filtered with `analyze_cpu_memory.py`.

Requirements: python 3.10+

In [1]:
import pandas as pd
import os
import matplotlib.pyplot as plt
import re
import numpy as np
from scipy.interpolate import make_interp_spline
from colorama import Fore
from colorama import init
init(autoreset=True)

import pprint

plt.rcParams.update({
    "text.usetex": True,
    "font.family": "sans-serif",
    "font.sans-serif": "Helvetica",
    "font.size" : 10
})

plt.rc('text.latex', preamble=r'\usepackage[helvet]{sfmath}')

In [2]:
palette = {
    "sensing"       : "#bda5c9", # pink
    "localization"  : "#96bcf3", # blue
    "perception"    : "#a6cba4", # green
    "planning"      : "#fcde83", # yellow
    "control"       : "#fac185", # orange
    "map"           : "#8da3b4", # blueish gray
    "system"        : "#cfcfcf", # gray
    "other"         : "#7bccd1", # turquoise
    "simulation"    : "#ffcdcd", # red
    "rviz"          : "#b691fc", # purple

    "bare-metal"    : "#c8ced4", # grey
    "docker"        : "#4c84f1", # blue
    "k3s"           : "#fdd56d"  # yellow
}

# Define the KPIs (Key Performance Indicators)
columns = ['CPU', 'RAM']

curve_names = {"rviz" : "RViz", "other" : "Vehicle\nInterface"}

configs = ["bare-metal", "docker", "k3s"]

page_width = 6.10356 # inches

In [3]:
# Specify the data and results folder paths
data_folder = 'data'
results_folder = 'results'

# Create the results folder if it doesn't exist
os.makedirs(results_folder, exist_ok=True)

# Get list of experiments (assuming each experiment has its own folder under the data folder)
experiments = sorted([dir for dir in os.listdir(data_folder) if os.path.isdir(os.path.join(data_folder, dir)) and os.path.exists(os.path.join(data_folder, dir, "cpu_mem_analysis")) and "ignore" not in dir])

### Load timeline data

Regardless of whether you want to **plot the timeline** or you only care about the average, you need to load the timeline data first.

In [4]:
# Create dictionaries to store timeline data
timeline_data = {}

# Iterate over the experiments
for experiment in experiments:
    print(f"{Fore.GREEN}Starting for {experiment}")

    if experiment not in timeline_data:
        timeline_data[experiment] = []

    # hardcode folders to enforce order
    modules = ["sensing", "localization", "perception", "planning", "control", "map", "simulation", "system", "rviz", "other"]
    # enforce "other" being processed first
    modules.reverse()
    for i, module in enumerate(modules):
        cpu_path = os.path.join(data_folder, experiment, 'cpu_mem_analysis', module, 'CPU_sums.csv')
        mem_path = os.path.join(data_folder, experiment, 'cpu_mem_analysis', module, 'RAM_sums.csv')

        df_cpu = pd.read_csv(cpu_path, header=None)
        df_mem = pd.read_csv(mem_path, header=None)

        df_cpu.columns = ['CPU']
        df_mem.columns = ['RAM']

        df_duration = pd.read_csv(os.path.join(data_folder, experiment, 'duration.txt'))
        seconds_duration = df_duration["seconds"].iloc[0]
        # timestamps are not at 1 Hz so we normalize them to visualize over seconds
        timestamps_to_seconds_factor = df_duration["timestamps"].iloc[0] / seconds_duration

        cpu_index_list = df_cpu[df_cpu['CPU'].gt(0.0)].index
        mem_index_list = df_mem[df_mem['RAM'].gt(0.0)].index
        if len(cpu_index_list) == 0 and len(mem_index_list) == 0:
            continue
        elif len(cpu_index_list) == 0 or len(mem_index_list) == 0:
            raise ValueError(f'CPU or Memory usage is 0 but not both for "{module}"')

        # first, last, and global_first timestamps are used to trim the process data to its active time
        cpu_first_timestamp = cpu_index_list[0]
        mem_first_timestamp = mem_index_list[0]
        first_timestamp = min(cpu_first_timestamp, mem_first_timestamp)
        # assume "other" always start first
        if i == 0:
            global_first_timestamp = first_timestamp
        else:
            # check the assumption
            if first_timestamp < global_first_timestamp:
                component = ""
                if cpu_first_timestamp < global_first_timestamp:
                    component = "CPU"
                if mem_first_timestamp < global_first_timestamp:
                    if component != "":
                        component += " and RAM"
                    else:
                        component = "RAM"
                print(f'{Fore.RED}"{module}" started {component} {global_first_timestamp - first_timestamp} timestamps earlier than "other"')
                first_timestamp = global_first_timestamp

        last_timestamp = max(cpu_index_list[-1], mem_index_list[-1])

        df_cpu['Timestamp'] = df_cpu.index - global_first_timestamp
        df_mem['Timestamp'] = df_mem.index - global_first_timestamp

        df = pd.merge(df_cpu, df_mem, on='Timestamp')
        df = df[df.index >= first_timestamp]
        df = df[df.index <= last_timestamp]
        df['Timestamp'] /= timestamps_to_seconds_factor

        df['Experiment'] = experiment
        df['Module'] = module
        df['Duration'] = (last_timestamp - first_timestamp) / timestamps_to_seconds_factor
        timeline_data[experiment].append(df)

    # enforce "other" being plotted last (entirely for aesthetic purposes)
    timeline_data[experiment].reverse()

# Concatenate all experiment dataframes
combined_df = pd.concat([df for exp in timeline_data.values() for df in exp], ignore_index=True)

Starting for bare-metal-planning-0
Starting for bare-metal-planning-1
Starting for bare-metal-planning-2
Starting for bare-metal-planning-3
Starting for bare-metal-planning-4
Starting for bare-metal-rosbag-0
Starting for bare-metal-rosbag-1
Starting for bare-metal-rosbag-2
Starting for bare-metal-rosbag-3
Starting for bare-metal-rosbag-4
Starting for docker-planning-0
Starting for docker-planning-1
Starting for docker-planning-2
Starting for docker-planning-3
Starting for docker-planning-4
Starting for docker-rosbag-0
Starting for docker-rosbag-1
Starting for docker-rosbag-2
Starting for docker-rosbag-3
Starting for docker-rosbag-4
Starting for k3s-planning-0
Starting for k3s-planning-1
Starting for k3s-planning-2
Starting for k3s-planning-3
Starting for k3s-planning-4
Starting for k3s-rosbag-0
Starting for k3s-rosbag-1
Starting for k3s-rosbag-2
Starting for k3s-rosbag-3
Starting for k3s-rosbag-4
Starting for wcm-k3s-rosbag


### Calculate and store statistics for each experiment
Also needed for all further cells.

In [5]:
experiment_data = {}

# Perform analysis for each KPI
for column in columns:
    for experiment in timeline_data.keys():
        # Get the dataframe for this experiment
        dfs = pd.concat(timeline_data[experiment], ignore_index=True)
        for module in dfs['Module'].unique():
            df_experiment = dfs[dfs['Module'] == module]

            # Calculate statistical measures
            mean = df_experiment[column].mean()
            std = df_experiment[column].std()
            minimum = df_experiment[column].min()
            maximum = df_experiment[column].max()

            # Create a DataFrame for the statistical measures
            stats_data = {
                'Experiment': experiment,
                'KPI': column,
                'Mean': mean,
                'Standard Deviation': std,
                'Minimum': minimum,
                'Maximum': maximum
            }
            if "bare-metal" in experiment:
                config = "bare-metal"
            elif "docker" in experiment:
                config = "docker"
            elif "k3s" in experiment:
                config = "k3s"
            else:
                raise ValueError(f"Unknown config in {experiment}")
    
            if "rosbag" in experiment:
                simulation = "rosbag"
            elif "planning" in experiment:
                simulation = "planning"
            else:
                raise ValueError(f"Unknown simulation in {experiment}")
            
            if column not in experiment_data:
                experiment_data[column] = {}
            if simulation not in experiment_data[column]:
                experiment_data[column][simulation] = {}
            if config not in experiment_data[column][simulation]:
                experiment_data[column][simulation][config] = {}
            if module not in experiment_data[column][simulation][config]:
                experiment_data[column][simulation][config][module] = pd.DataFrame(columns=["Experiment", "Util"])
            experiment_data[column][simulation][config][module].loc[len(experiment_data[column][simulation][config][module])] = [experiment, mean]
            stats_df = pd.DataFrame([stats_data])

            # Save the statistical measures to a CSV file
            os.makedirs(os.path.join(results_folder, experiment, 'stats'), exist_ok=True)
            results_filename = os.path.join(results_folder, experiment, 'stats', f'{module}_{column}.csv')
            stats_df.to_csv(results_filename, index=False)

pprint.pprint(experiment_data)

{'CPU': {'planning': {'bare-metal': {'control':               Experiment       Util
0  bare-metal-planning-0   5.056502
1  bare-metal-planning-1  11.988708
2  bare-metal-planning-2  11.772789
3  bare-metal-planning-3  12.831873
4  bare-metal-planning-4  13.722951,
                                     'map':               Experiment      Util
0  bare-metal-planning-0  3.644619
1  bare-metal-planning-1  4.270593
2  bare-metal-planning-2  5.795238
3  bare-metal-planning-3  4.415936
4  bare-metal-planning-4  4.488163,
                                     'other':               Experiment       Util
0  bare-metal-planning-0  66.177223
1  bare-metal-planning-1  76.910174
2  bare-metal-planning-2  95.354839
3  bare-metal-planning-3  83.924710
4  bare-metal-planning-4  85.773413,
                                     'perception':               Experiment      Util
0  bare-metal-planning-0  3.250673
1  bare-metal-planning-1  4.486596
2  bare-metal-planning-2  5.291837
3  bare-metal-planning-3  

                               'localization':         Experiment       Util
0  docker-rosbag-0  37.580091
1  docker-rosbag-1  38.154854
2  docker-rosbag-2  36.961745
3  docker-rosbag-3  40.346259
4  docker-rosbag-4  37.900685,
                               'map':         Experiment      Util
0  docker-rosbag-0  7.628875
1  docker-rosbag-1  7.657870
2  docker-rosbag-2  7.286667
3  docker-rosbag-3  8.346259
4  docker-rosbag-4  7.979452,
                               'other':         Experiment        Util
0  docker-rosbag-0  104.899556
1  docker-rosbag-1  102.214458
2  docker-rosbag-2   99.880519
3  docker-rosbag-3  114.354667
4  docker-rosbag-4  113.810000,
                               'perception':         Experiment       Util
0  docker-rosbag-0  53.500608
1  docker-rosbag-1  51.770370
2  docker-rosbag-2  51.976667
3  docker-rosbag-3  61.244218
4  docker-rosbag-4  55.324490,
                               'planning':         Experiment       Util
0  docker-rosbag-0  34.238356
1  

4  k3s-planning-4   0.3,
                              'other':        Experiment      Util
0  k3s-planning-0  0.946058
1  k3s-planning-1  1.027489
2  k3s-planning-2  1.132000
3  k3s-planning-3  1.017763
4  k3s-planning-4  1.061310,
                              'perception':        Experiment      Util
0  k3s-planning-0  0.100000
1  k3s-planning-1  0.099782
2  k3s-planning-2  0.100000
3  k3s-planning-3  0.100000
4  k3s-planning-4  0.100000,
                              'planning':        Experiment      Util
0  k3s-planning-0  3.192469
1  k3s-planning-1  3.378384
2  k3s-planning-2  2.606757
3  k3s-planning-3  2.726490
4  k3s-planning-4  3.087952,
                              'rviz':        Experiment      Util
0  k3s-planning-0  1.475941
1  k3s-planning-1  1.479694
2  k3s-planning-2  1.470946
3  k3s-planning-3  1.478667
4  k3s-planning-4  1.480120,
                              'simulation':        Experiment      Util
0  k3s-planning-0  0.100000
1  k3s-planning-1  0.099782
2  k3s-p

### Plot utilization over time
This cell takes quite some time to execute and creates 2 plots for each experiement. Therefore, better skip this cell of not interested.

In [6]:
for column in columns:
    for experiment in timeline_data.keys():
        fig, ax = plt.subplots(layout='constrained')
        fig.set_figwidth(page_width)
        fig.set_figheight(page_width / 2)
        ax.get_yaxis().get_major_formatter().set_useOffset(False)
        for experiment_df in timeline_data[experiment]:
            module = experiment_df['Module'].iloc[0]
            label = curve_names[module] if module in curve_names else module.capitalize()

            x = experiment_df['Timestamp'].values
            y = experiment_df[column].values
            spline = make_interp_spline(x, y)
            duration = np.floor(experiment_df['Duration'].iloc[0]).astype(int)
            x_interp = np.linspace(x.min(), x.max(), duration)
            y_interp = spline(x_interp)
            ax.plot(x_interp, y_interp, label=label, color=palette[module])
        ax.set_xlabel('Seconds')
        ax.set_ylabel(f"{column} utilization in \%")
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.spines['bottom'].set_visible(False)
        ax.spines['left'].set_visible(False)
        ax.legend()
        ax.set_axisbelow(True)
        ax.yaxis.grid(True)
        ax.xaxis.grid(True)
        ax.tick_params(left=False, bottom=False)
        plt.xticks()
        plt.savefig(os.path.join(results_folder, experiment, f'{column}_timeaxis.pdf'))
        plt.close(fig)

TypeError: __init__() got an unexpected keyword argument 'layout'

### Extract data used for barplots of total utilization

Calculate total usage, its deviation during each run, and usage by container processes.

In [7]:
total_data = {}

for experiment in experiments:

    if "bare-metal" in experiment:
        config = "bare-metal"
    elif "docker" in experiment:
        config = "docker"
    elif "k3s" in experiment:
        config = "k3s"
    else:
        raise ValueError(f"Unknown config in {experiment}")
    
    if "rosbag" in experiment:
        simulation = "rosbag"
    elif "planning" in experiment:
        simulation = "planning"
    else:
        raise ValueError(f"Unknown simulation in {experiment}")

    for metric in columns:

        if simulation not in total_data:
            total_data[simulation] = {}
        if  metric not in total_data[simulation]:
            total_data[simulation][metric] = {}
        if config not in total_data[simulation][metric]:
            total_data[simulation][metric][config] = {"mean" : [], "mean_cont" : [], "var" : []}

        # autoware util
        df_total_no_container = pd.read_csv(os.path.join(data_folder, experiment, 'cpu_mem_analysis', "total", f'{metric}_sums.csv'), header=0)
        df_total_no_container.columns = [metric]


        if config != "bare-metal":
            # autoware + container
            df_total_container = pd.read_csv(os.path.join(data_folder, experiment, 'cpu_mem_analysis', "container", f'{metric}_sums.csv'), header=0)
            df_total_container.columns = [metric]
            # only the segment when autoware is active is relevant
            df_total_container = df_total_container[df_total_no_container[metric].gt(0.0)]
            df_total_no_container = df_total_no_container[df_total_no_container[metric].gt(0.0)]
            var = df_total_container[metric].var()
            series_container = df_total_container[metric] - df_total_no_container[metric]
            mean_container = series_container.mean()
        else:
            df_total_no_container = df_total_no_container[df_total_no_container[metric].gt(0.0)]
            var = df_total_no_container[metric].var()
            mean_container = 0

        mean = df_total_no_container[metric].mean()

        total_data[simulation][metric][config]["mean"].append(mean)
        total_data[simulation][metric][config]["var"].append(var)
        total_data[simulation][metric][config]["mean_cont"].append(mean_container)

pprint.pprint(total_data)

{'planning': {'CPU': {'bare-metal': {'mean': [202.63210412147504,
                                              258.3755520504732,
                                              293.8090322580646,
                                              270.41930501930506,
                                              281.66984126984136],
                                     'mean_cont': [0, 0, 0, 0, 0],
                                     'var': [20186.386619258705,
                                             18687.280901431383,
                                             36684.536930875576,
                                             16380.394742150791,
                                             25627.141875671925]},
                      'docker': {'mean': [291.9631578947368,
                                          280.1048267326733,
                                          316.2338129496403,
                                          320.7931297709923,
                                 

                                   'var': [31.776112604637948,
                                           29.80091749651295,
                                           35.12060997067449,
                                           21.618766445916116,
                                           19.013952572706934]},
                    'docker': {'mean': [22.283703703703704,
                                        22.26742081447964,
                                        22.16168831168831,
                                        22.506666666666668,
                                        22.371999999999996],
                               'mean_cont': [0.3013333333333378,
                                             0.5000000000000034,
                                             0.3000000000000047,
                                             0.3026666666666711,
                                             0.30466666666667097],
                               'var': [18.24210515441258,
 

Calculate deviation of mean values across runs of the same configuration. Print if above 5%. This is **not** the deviation that will be visualized in the plot!

In [8]:
for sim, sim_data in total_data.items():
    for metric, metric_data in sim_data.items():
        for config, config_data in metric_data.items():
            mean = np.mean(config_data['mean'])
            std = np.std(config_data['mean'])
            std_p = std / mean * 100
            if std_p > 5.0:
                print(f'{sim} {metric} {config} has std = {std_p}% of mean')

planning CPU bare-metal has std = 12.104833859212679% of mean
planning CPU docker has std = 5.995715591028161% of mean
rosbag CPU k3s has std = 31.870801909284687% of mean


### Print total utilization
CPU and RAM are plotted as subplots in the same figure, one figure per simulation type.

In [9]:
from matplotlib.patches import Patch

# darker palette for docker and k3s to visualize the fraction of container processes
container_palette = {
    "bare-metal"    : "#ffffff", # white
    "docker"        : "#0d41a6", # darker blue
    "k3s"           : "#dea103"  # darker yellow
}

for sim, sim_data in total_data.items():

    fig, ax = plt.subplots(ncols=len(sim_data), nrows=1, layout='constrained')
    fig.set_figwidth(page_width)
    fig.set_figheight(page_width / 2)

    for i, (metric, metric_data) in enumerate(sim_data.items()):

        metric_data = dict(sorted(metric_data.items(), key=lambda item : configs.index(item[0])))

        mean_values = [np.mean(v["mean"]) for v in metric_data.values()]
        mean_cont_values = [np.mean(v["mean_cont"]) for v in metric_data.values()]

        # mean std is calculated as sqrt of mean variance
        std_values = [np.sqrt(np.mean(v["var"])) for v in metric_data.values()]

        x_len = len(metric_data)
        x = np.arange(x_len)
        width = 0.5

        rects = ax[i].bar(x, mean_values, width, color=[palette[c] for c in metric_data.keys()])
        
        rects_container = ax[i].bar(x, mean_cont_values, width, bottom=mean_values,
                        yerr=std_values,
                        error_kw=dict(lw=1, capsize=7, capthick=1), color=[container_palette[c] for c in metric_data.keys()])

        ax[i].set_ylabel(f"{metric} utilization in \%")
        if sim == "rosbag":
            ax[i].set_ybound(upper= 800 if metric == "CPU" else 30)
        ax[i].set_xticks(x, [c.capitalize() for c in metric_data.keys()])
        ax[i].spines['top'].set_visible(False)
        ax[i].spines['right'].set_visible(False)
        ax[i].spines['bottom'].set_visible(False)
        ax[i].spines['left'].set_visible(False)
        ax[i].set_axisbelow(True)
        ax[i].yaxis.grid(True)
        ax[i].tick_params(left=False, bottom=False)

    pa1 = Patch(facecolor=container_palette["docker"], edgecolor=container_palette["docker"])
    pa2 = Patch(facecolor=container_palette["k3s"], edgecolor=container_palette["k3s"])
    fig.legend(handles=[pa1, pa2], labels=['', 'Container processes'], ncol=2, columnspacing=0, loc='upper center')
    
    plt.savefig(os.path.join(results_folder, f'{sim}_total.pdf'))
    plt.close(fig)

TypeError: __init__() got an unexpected keyword argument 'layout'

### Print utilization per component

In [10]:
for metric, metric_data in experiment_data.items():
    for sim, sim_data in metric_data.items():
        number_modules_per_sim = [len(conf) for conf in sim_data.values()]
        number_modules = number_modules_per_sim[0]
        assert number_modules_per_sim.count(number_modules) == len(number_modules_per_sim)
        x = np.arange(number_modules)
        width = 1 / (len(sim_data.values()) + 1)
        multiplier = 0

        fig, ax = plt.subplots(layout='constrained')
        if "rosbag" in sim:
            fig.set_figwidth(7.7)
            fig.set_figheight(7.7 / 2)
        else:
            fig.set_figwidth(page_width)
            fig.set_figheight(page_width / 2)

        sim_data = dict(sorted(sim_data.items(), key=lambda item : configs.index(item[0])))

        for config, measurement in sim_data.items():
            offset = width * multiplier
            values = [np.mean(module_data["Util"]) for module_data in measurement.values()]
            rects = ax.bar(x + offset, values, width * 0.85, label=config.capitalize(), color=palette[config])
            ax.bar_label(rects, padding=2, fontsize=7,fmt=lambda x: f'{x:0.1f}' if x < 100.0 else f'{x:0.0f}')
            multiplier += 1

        ax.set_ylabel(f"{metric} utilization in \%")
        ax.set_xticks(x + width * (len(sim_data.values()) - 1) / 2, [m.capitalize() if m not in curve_names else curve_names[m] for m in (measurement.keys())])
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.spines['bottom'].set_visible(False)
        ax.spines['left'].set_visible(False)
        ax.legend()
        ax.set_axisbelow(True)
        ax.yaxis.grid(True)
        ax.tick_params(left=False, bottom=False)
        plt.xticks()
        plt.savefig(os.path.join(results_folder, f'{sim}_{metric}_component.pdf'))
        plt.close(fig)


TypeError: __init__() got an unexpected keyword argument 'layout'